# Input data validation

Check the sensor-location CSV against the required schema and known ENV sensor values before preprocessing. The ID may be written in scientific notation in the CSV, so validation compares its numeric value.

## Engineering documentation (project brief alignment)

The MMA3001 project brief requires the project documentation to explain the engineering problem and intended use, define input/output formats and domains, describe how invalid inputs are handled, explain and justify the computational method, and report validation evidence, limitations and lessons learned. The final project report must also cover comparison/optimisation, AI-use reflection and relevant references. The repository is expected to include a README, HTML source-code documentation, source code with comments/docstrings, and test functions with a test report. See [the project brief](./MMA3001_2026_Project_Brief.pdf).

### This validation component

**Purpose and intended use.** Check the ENV sensor-location mapping before it is used in preprocessing, so incompatible identifiers are reported early. This is an input-contract check, not a cleaning or transformation step.

**Input format and domain.** The notebook reads `Sensor ID and Locations - ENV.csv` from the project root as a UTF-8 CSV with a header row and at least one data row. The validator accepts a pandas `DataFrame`; it requires `Sensor` and `ID` columns and permits additional columns, which are not checked here. `Sensor` is expected to be the exact category `Env`. `ID` is expected to be the numeric identifier `6012000000000`; scientific notation such as `6.012E+12` represents the same value. These fields are identifiers, not measured physical quantities, so no measurement units apply.

**Output.** On success, validation returns `None` and the test suite passes; it does not modify the input or produce a transformed data file. On failure, it raises `ValueError` identifying a missing required column, empty input, unexpected/null sensor category, or invalid/unexpected ID. A failing test suite also fails the notebook cell.

**Scope and limitations.** The mapping check validates only the `Sensor` and `ID` contract and non-empty input. Time-series checks validate observed invariant columns and report timestamp comparisons without treating ordinary differences as errors. They do not check JSON payload structure, measurement ranges, record uniqueness, or sensor-to-location relationships.

**Verification evidence.** The location tests check the supplied CSV and reject unexpected sensor and ID values, a missing column, and empty input. Time-series tests recheck invariant columns and timestamp comparisons from disk in chunks. Timestamp differences are reported for investigation, not treated as validation failures. Re-run this code cell after changing an input or its contract, and retain the test report as validation evidence.

**Observed time-series constants.** Every environmental row has `sensorid = 6.012E+12`, `devicetype = env`, `status = Processed`, and `processing_errors = NULL` (the literal CSV text). Every occupancy row has `deviceid = 56923747-b390-4f2a-8930-808429132266`. Sensor IDs are compared numerically so equivalent integer and scientific-notation representations pass. Other fields vary and are not constrained by these invariant checks.

**Time-series input/output behavior.** The input files are `5EnvSensor_MayToDec2024_180kRows.csv` and `5occupancySensor_MayToDec2024_9MRows.csv`, headered CSVs read as strings in 100,000-row chunks. Successful constant validation returns `None` without modifying either file. Missing invariant columns, empty input, or an unexpected constant raise `ValueError`; CSV parsing errors are surfaced by pandas. The identifier and status fields have no measurement units.

**Observed relationships.** Environmental `createdate` and `processdate` matched in all 179,447 rows. Occupancy `collecteddate` and `occupancystatuschangedate` matched in 1,106,003 of 9,091,732 rows (12.16%).

### Project-level documentation still required

This notebook documents only the validation component. The final project documentation and repository still need to address the full brief: the engineering context/problem and intended use; top-level project inputs and outputs; computational approach and its justification; validation method and results; comparison/optimisation; limitations and lessons learned; AI-use reflection; references; and repository-level README, generated HTML source documentation, licensing, and test report. Add evidence as those project decisions and results become available; do not infer them from this input check.

In [7]:
from pathlib import Path
import unittest

import pandas as pd


CSV_PATH = Path("Sensor ID and Locations - ENV.csv")
EXPECTED_SENSOR = "Env"
EXPECTED_ID = 6_012_000_000_000


def validate_env_sensor_locations(data: pd.DataFrame) -> None:
    """Validate the required ENV sensor-location columns and values.

    Args:
        data: Mapping rows with `Sensor` and `ID` columns. IDs may be
            represented numerically or in scientific notation.

    Raises:
        ValueError: If required columns or rows are missing, or any sensor
            category or identifier is outside the expected ENV contract.
    """
    missing_columns = {"Sensor", "ID"} - set(data.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {', '.join(sorted(missing_columns))}")
    if data.empty:
        raise ValueError("The sensor-location file must contain at least one row")
    if data["Sensor"].isna().any() or not data["Sensor"].eq(EXPECTED_SENSOR).all():
        raise ValueError(f'Every Sensor value must be "{EXPECTED_SENSOR}"')

    ids = pd.to_numeric(data["ID"], errors="coerce")
    if ids.isna().any() or not ids.eq(EXPECTED_ID).all():
        raise ValueError(f"Every ID value must equal {EXPECTED_ID}")


ENV_CSV_PATH = Path("5EnvSensor_MayToDec2024_180kRows.csv")
OCCUPANCY_CSV_PATH = Path("5occupancySensor_MayToDec2024_9MRows.csv")
EXPECTED_ENV_CONSTANTS = {
    "sensorid": EXPECTED_ID,
    "devicetype": "env",
    "status": "Processed",
    "processing_errors": "NULL",
}
EXPECTED_OCCUPANCY_CONSTANTS = {
    "deviceid": "56923747-b390-4f2a-8930-808429132266",
}
OCCUPANCY_COLUMNS_TO_EXCLUDE = {
    "deviceid",
    "occupancystatus",
    "previousoccupancystatus",
}
TIMESTAMP_PAIRS = {
    "Environmental sensor data": (ENV_CSV_PATH, "createdate", "processdate"),
    "Occupancy sensor data": (
        OCCUPANCY_CSV_PATH,
        "collecteddate",
        "occupancystatuschangedate",
    ),
}


def validate_constant_columns(
    data: pd.DataFrame,
    expected_constants: dict[str, object],
    dataset_name: str,
) -> None:
    """Validate constant field values in one non-empty data chunk.

    Args:
        data: CSV rows with values read as strings.
        expected_constants: Required column names and expected values. Numeric
            expectations are compared numerically; other expectations exactly.
        dataset_name: Dataset label included in validation errors.

    Raises:
        ValueError: If required columns are absent, the chunk is empty, or any
            value differs from its expected constant.
    """
    missing_columns = set(expected_constants) - set(data.columns)
    if missing_columns:
        raise ValueError(
            f"{dataset_name}: missing required columns: {', '.join(sorted(missing_columns))}"
        )
    if data.empty:
        raise ValueError(f"{dataset_name}: input must contain at least one row")

    for column, expected_value in expected_constants.items():
        actual_values = data[column]
        if isinstance(expected_value, (int, float)):
            actual_values = pd.to_numeric(actual_values, errors="coerce")
        if not actual_values.eq(expected_value).all():
            raise ValueError(
                f"{dataset_name}: every {column} value must equal {expected_value!r}"
            )


def validate_csv_constant_columns(
    csv_path: Path,
    expected_constants: dict[str, object],
    dataset_name: str,
    chunksize: int = 100_000,
) -> None:
    """Validate configured constant columns in a CSV using bounded memory.

    Only the requested columns are read. A numeric expectation, such as a
    sensor ID, is compared numerically so scientific notation is accepted.

    Raises:
        ValueError: If required columns are missing or the CSV has no rows.
        pandas.errors.ParserError: If pandas cannot parse the CSV.
    """
    headers = pd.read_csv(csv_path, nrows=0, dtype=str, keep_default_na=False).columns
    missing_columns = set(expected_constants) - set(headers)
    if missing_columns:
        raise ValueError(
            f"{dataset_name}: missing required columns: {', '.join(sorted(missing_columns))}"
        )

    row_count = 0
    for chunk in pd.read_csv(
        csv_path,
        usecols=list(expected_constants),
        dtype=str,
        keep_default_na=False,
        chunksize=chunksize,
    ):
        validate_constant_columns(chunk, expected_constants, dataset_name)
        row_count += len(chunk)

    if row_count == 0:
        raise ValueError(f"{dataset_name}: input must contain at least one row")


def count_timestamp_pair_equality(
    data: pd.DataFrame,
    first_column: str,
    second_column: str,
    dataset_name: str,
) -> dict[str, int | float]:
    """Count equal, different, and unparseable timestamp pairs in one chunk.

    Compare parsed UTC instants, so equal moments with different UTC offsets
    count as equal. Differences are reported for exploration, not rejected.
    """
    required_columns = {first_column, second_column}
    missing_columns = required_columns - set(data.columns)
    if missing_columns:
        raise ValueError(
            f"{dataset_name}: missing timestamp columns: {', '.join(sorted(missing_columns))}"
        )
    if data.empty:
        raise ValueError(f"{dataset_name}: input must contain at least one row")

    first = pd.to_datetime(data[first_column], errors="coerce", utc=True)
    second = pd.to_datetime(data[second_column], errors="coerce", utc=True)
    parseable = first.notna() & second.notna()
    equal_rows = int((parseable & first.eq(second)).sum())
    unequal_rows = int((parseable & first.ne(second)).sum())
    unparseable_rows = int((~parseable).sum())
    parseable_rows = equal_rows + unequal_rows
    return {
        "rows": len(data),
        "equal_rows": equal_rows,
        "unequal_rows": unequal_rows,
        "unparseable_rows": unparseable_rows,
        "match_rate": equal_rows / parseable_rows if parseable_rows else 0.0,
    }


def summarize_csv_timestamp_pair(
    csv_path: Path,
    first_column: str,
    second_column: str,
    dataset_name: str,
    chunksize: int = 100_000,
) -> dict[str, int | float]:
    """Summarize timestamp-pair equality across a CSV without loading it all."""
    headers = pd.read_csv(csv_path, nrows=0, dtype=str, keep_default_na=False).columns
    missing_columns = {first_column, second_column} - set(headers)
    if missing_columns:
        raise ValueError(
            f"{dataset_name}: missing timestamp columns: {', '.join(sorted(missing_columns))}"
        )

    totals = {"rows": 0, "equal_rows": 0, "unequal_rows": 0, "unparseable_rows": 0}
    for chunk in pd.read_csv(
        csv_path,
        usecols=[first_column, second_column],
        dtype=str,
        keep_default_na=False,
        chunksize=chunksize,
    ):
        counts = count_timestamp_pair_equality(
            chunk, first_column, second_column, dataset_name
        )
        for key in totals:
            totals[key] += int(counts[key])

    if totals["rows"] == 0:
        raise ValueError(f"{dataset_name}: input must contain at least one row")

    parseable_rows = totals["equal_rows"] + totals["unequal_rows"]
    totals["match_rate"] = totals["equal_rows"] / parseable_rows if parseable_rows else 0.0
    return totals


class TestEnvSensorLocations(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.data = pd.read_csv(CSV_PATH)

    def test_current_file_matches_expected_values(self):
        validate_env_sensor_locations(self.data)

    def test_unexpected_sensor_value_is_rejected(self):
        invalid_data = self.data.copy()
        invalid_data.loc[invalid_data.index[0], "Sensor"] = "Occupancy"
        with self.assertRaisesRegex(ValueError, "Sensor"):
            validate_env_sensor_locations(invalid_data)

    def test_unexpected_id_value_is_rejected(self):
        invalid_data = self.data.copy()
        invalid_data.loc[invalid_data.index[0], "ID"] = 123
        with self.assertRaisesRegex(ValueError, "ID"):
            validate_env_sensor_locations(invalid_data)

    def test_missing_required_column_is_rejected(self):
        invalid_data = self.data.drop(columns=["ID"])
        with self.assertRaisesRegex(ValueError, "Missing required columns"):
            validate_env_sensor_locations(invalid_data)

    def test_empty_file_is_rejected(self):
        with self.assertRaisesRegex(ValueError, "at least one row"):
            validate_env_sensor_locations(self.data.iloc[0:0])


class TestSensorTimeSeriesConstants(unittest.TestCase):
    def test_environment_file_constants(self):
        validate_csv_constant_columns(
            ENV_CSV_PATH, EXPECTED_ENV_CONSTANTS, "Environmental sensor data"
        )

    def test_occupancy_file_constants(self):
        validate_csv_constant_columns(
            OCCUPANCY_CSV_PATH, EXPECTED_OCCUPANCY_CONSTANTS, "Occupancy sensor data"
        )

    def test_occupancy_reduced_read_columns_exist(self):
        headers = set(pd.read_csv(OCCUPANCY_CSV_PATH, nrows=0).columns)
        missing_columns = OCCUPANCY_COLUMNS_TO_EXCLUDE - headers
        self.assertFalse(missing_columns, f"Missing selected columns: {sorted(missing_columns)}")

    def test_each_environment_constant_is_enforced(self):
        valid_data = pd.DataFrame([
            {column: str(value) for column, value in EXPECTED_ENV_CONSTANTS.items()}
            for _ in range(2)
        ])
        for column in EXPECTED_ENV_CONSTANTS:
            with self.subTest(column=column):
                invalid_data = valid_data.copy()
                invalid_data.loc[0, column] = "unexpected"
                with self.assertRaisesRegex(ValueError, column):
                    validate_constant_columns(
                        invalid_data, EXPECTED_ENV_CONSTANTS, "Environmental sensor data"
                    )

    def test_scientific_notation_sensor_id_is_accepted(self):
        data = pd.DataFrame([
            {column: str(value) for column, value in EXPECTED_ENV_CONSTANTS.items()}
        ])
        data.loc[0, "sensorid"] = "6.012E+12"
        validate_constant_columns(data, EXPECTED_ENV_CONSTANTS, "Environmental sensor data")

    def test_unexpected_occupancy_device_id_is_rejected(self):
        data = pd.DataFrame({"deviceid": ["unexpected"]})
        with self.assertRaisesRegex(ValueError, "deviceid"):
            validate_constant_columns(data, EXPECTED_OCCUPANCY_CONSTANTS, "Occupancy sensor data")

    def test_missing_constant_column_is_rejected(self):
        data = pd.DataFrame({"devicetype": ["env"]})
        with self.assertRaisesRegex(ValueError, "missing required columns"):
            validate_constant_columns(data, EXPECTED_ENV_CONSTANTS, "Environmental sensor data")

    def test_empty_chunk_is_rejected(self):
        data = pd.DataFrame(columns=EXPECTED_OCCUPANCY_CONSTANTS)
        with self.assertRaisesRegex(ValueError, "at least one row"):
            validate_constant_columns(data, EXPECTED_OCCUPANCY_CONSTANTS, "Occupancy sensor data")


class TestTimestampPairEquality(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.file_summaries = {
            dataset_name: summarize_csv_timestamp_pair(
                csv_path, first_column, second_column, dataset_name
            )
            for dataset_name, (csv_path, first_column, second_column) in TIMESTAMP_PAIRS.items()
        }

    def test_equal_and_unequal_pairs_are_counted_without_rejection(self):
        data = pd.DataFrame({
            "first": ["2024-01-01T00:00:00+00:00", "2024-01-01T00:00:00+00:00"],
            "second": ["2024-01-01T11:00:00+11:00", "2024-01-01T00:00:01+00:00"],
        })
        result = count_timestamp_pair_equality(data, "first", "second", "Synthetic data")
        self.assertEqual(result["equal_rows"], 1)
        self.assertEqual(result["unequal_rows"], 1)
        self.assertEqual(result["unparseable_rows"], 0)
        self.assertEqual(result["match_rate"], 0.5)

    def test_unparseable_pairs_are_reported(self):
        data = pd.DataFrame({"first": ["", "not-a-date"], "second": ["2024-01-01", "2024-01-02"]})
        result = count_timestamp_pair_equality(data, "first", "second", "Synthetic data")
        self.assertEqual(result["unparseable_rows"], 2)
        self.assertEqual(result["match_rate"], 0.0)

    def test_current_csv_pairs_are_fully_counted(self):
        for dataset_name, result in self.file_summaries.items():
            with self.subTest(dataset=dataset_name):
                self.assertGreater(result["rows"], 0)
                self.assertEqual(
                    result["rows"],
                    result["equal_rows"] + result["unequal_rows"] + result["unparseable_rows"],
                )


test_loader = unittest.defaultTestLoader
test_suite = unittest.TestSuite([
    test_loader.loadTestsFromTestCase(TestEnvSensorLocations),
    test_loader.loadTestsFromTestCase(TestSensorTimeSeriesConstants),
    test_loader.loadTestsFromTestCase(TestTimestampPairEquality),
])
test_result = unittest.TextTestRunner(verbosity=2).run(test_suite)
assert test_result.wasSuccessful(), "Input data validation tests failed"

for dataset_name, result in TestTimestampPairEquality.file_summaries.items():
    print(
        f"{dataset_name}: {result['equal_rows']:,}/{result['rows']:,} equal "
        f"({result['match_rate']:.2%}); {result['unequal_rows']:,} unequal; "
        f"{result['unparseable_rows']:,} missing/unparseable"
    )


test_current_file_matches_expected_values (__main__.TestEnvSensorLocations.test_current_file_matches_expected_values) ... ok
test_empty_file_is_rejected (__main__.TestEnvSensorLocations.test_empty_file_is_rejected) ... ok
test_missing_required_column_is_rejected (__main__.TestEnvSensorLocations.test_missing_required_column_is_rejected) ... ok
test_unexpected_id_value_is_rejected (__main__.TestEnvSensorLocations.test_unexpected_id_value_is_rejected) ... ok
test_unexpected_sensor_value_is_rejected (__main__.TestEnvSensorLocations.test_unexpected_sensor_value_is_rejected) ... ok
test_each_environment_constant_is_enforced (__main__.TestSensorTimeSeriesConstants.test_each_environment_constant_is_enforced) ... ok
test_empty_chunk_is_rejected (__main__.TestSensorTimeSeriesConstants.test_empty_chunk_is_rejected) ... ok
test_environment_file_constants (__main__.TestSensorTimeSeriesConstants.test_environment_file_constants) ... ok
test_missing_constant_column_is_rejected (__main__.TestSensorTime

Environmental sensor data: 179,447/179,447 equal (100.00%); 0 unequal; 0 missing/unparseable
Occupancy sensor data: 1,106,003/9,091,732 equal (12.16%); 7,985,729 unequal; 0 missing/unparseable


## Pre-processing hypothesis: exclude invariant columns when loading data

The constant-column checks above establish data invariants; they do not change the source CSV files. A possible optimisation for later analysis is to load only the fields required for a specific task. With pandas, `read_csv(..., usecols=...)` can avoid constructing DataFrame columns that are not selected. This is preferable to reading every column and then copying the entire DataFrame just to drop columns, because the latter first loads the unnecessary data into memory.

For example, after deciding which fields an analysis actually needs:

```python
env_analysis_columns = ["id", "jsondata", "createdate", "processdate"]
env_data = pd.read_csv(
    ENV_CSV_PATH,
    usecols=env_analysis_columns,
)
```

This is only an example, not a final selection: `id`, payload, or timestamps may still be required for traceability or future joins. Choose columns based on the analysis requirements, and retain any invariant identifiers needed for interpretation or linking datasets. The validation cell should continue checking expected constants independently of whichever columns a later analysis loads.

### Proposed comparison

Treat reduced-column loading as a hypothesis to test, not an assumed improvement. Compare a normal full-column read with a `usecols` read using the same input file and environment. Record elapsed time and peak memory, and check that both reads yield the same row count and that every required analysis field is present. CSV input still has to be scanned, so performance gains depend on the parser and workload; measure rather than assume.

For the environmental CSV, the benchmark first compares `createdate` with `processdate` across every row. It excludes `processdate` only when every pair parses and represents the same instant. For occupancy, the reduced-read comparison omits `deviceid`, `occupancystatus`, and `previousoccupancystatus`. The device ID is separately checked as a constant in the validation cell; omitting the two status columns is an exploratory analytical choice, not a claim that they are invariant or interchangeable with `headcount`. Both reads preserve the same rows, and the original CSV is not modified. Assess whether the remaining fields support later analysis before adopting the reduced selection.

Keep the original CSV files unchanged as raw inputs. If repeated analysis justifies a reusable reduced dataset, write it to a separate derived file (Parquet is one option), and document its source, selected columns, row count, and transformation. Do not overwrite or silently replace the originals.

In [8]:
import time


def benchmark_csv_column_selection(
    csv_path: Path,
    columns_to_exclude: set[str],
    chunksize: int = 100_000,
    timestamp_pair: tuple[str, str] | None = None,
) -> dict[str, object]:
    """Compare full and reduced CSV reads using selected column exclusions.

    Reports elapsed time and the largest chunk DataFrame footprint; this is
    not a measurement of total process peak memory. The source file is read
    only and is never modified.

    Args:
        csv_path: Source CSV to benchmark.
        columns_to_exclude: Columns omitted from the reduced read. This can
            include validated invariants and exploratory analytical choices.
        chunksize: Number of rows read into each DataFrame chunk.
        timestamp_pair: Optional `(reference, candidate)` columns. The candidate
            is excluded only if every row has two parseable, equal timestamps.

    Returns:
        Row count, selected column count, elapsed time, and largest chunk
        memory for each read configuration.

    Raises:
        ValueError: If a requested exclusion column is absent or no columns remain.
        AssertionError: If the full and reduced reads have different row counts.
    """
    headers = list(pd.read_csv(csv_path, nrows=0).columns)
    missing_columns = columns_to_exclude - set(headers)
    if missing_columns:
        raise ValueError(
            f"{csv_path.name}: cannot exclude missing columns {sorted(missing_columns)}"
        )

    excluded_columns = set(columns_to_exclude)
    timestamp_check = None
    if timestamp_pair is not None:
        reference_column, candidate_column = timestamp_pair
        timestamp_check = summarize_csv_timestamp_pair(
            csv_path, reference_column, candidate_column, csv_path.name, chunksize
        )
        if timestamp_check["equal_rows"] == timestamp_check["rows"]:
            excluded_columns.add(candidate_column)

    selected_columns = [column for column in headers if column not in excluded_columns]
    if not selected_columns:
        raise ValueError(f"{csv_path.name}: no columns remain after applying exclusions")

    def measure_read(usecols: list[str] | None) -> dict[str, float | int]:
        start_time = time.perf_counter()
        row_count = 0
        largest_chunk_bytes = 0
        for chunk in pd.read_csv(csv_path, usecols=usecols, chunksize=chunksize):
            row_count += len(chunk)
            largest_chunk_bytes = max(
                largest_chunk_bytes,
                int(chunk.memory_usage(index=True, deep=True).sum()),
            )
        return {
            "rows": row_count,
            "columns": len(headers) if usecols is None else len(usecols),
            "elapsed_seconds": time.perf_counter() - start_time,
            "largest_chunk_mib": largest_chunk_bytes / (1024 ** 2),
        }

    full_read = measure_read(usecols=None)
    reduced_read = measure_read(usecols=selected_columns)
    assert full_read["rows"] == reduced_read["rows"], (
        f"Row count changed for {csv_path.name}: "
        f"{full_read['rows']} full vs {reduced_read['rows']} reduced"
    )
    return {
        "full_read": full_read,
        "reduced_read": reduced_read,
        "excluded_columns": sorted(excluded_columns),
        "timestamp_check": timestamp_check,
    }


benchmark_cases = [
    (ENV_CSV_PATH, set(EXPECTED_ENV_CONSTANTS), ("createdate", "processdate")),
    (OCCUPANCY_CSV_PATH, OCCUPANCY_COLUMNS_TO_EXCLUDE, None),
]

benchmark_results = {}
for csv_path, columns_to_exclude, timestamp_pair in benchmark_cases:
    results = benchmark_csv_column_selection(
        csv_path,
        columns_to_exclude,
        timestamp_pair=timestamp_pair,
    )
    benchmark_results[csv_path.name] = results
    print(f"\n{csv_path.name}")
    for read_type in ("full_read", "reduced_read"):
        metrics = results[read_type]
        print(
            f"  {read_type}: {metrics['rows']:,} rows, {metrics['columns']} columns, "
            f"{metrics['elapsed_seconds']:.2f} s, "
            f"largest chunk {metrics['largest_chunk_mib']:.2f} MiB"
        )
    print(f"  Excluded columns: {results['excluded_columns']}")
    if results["timestamp_check"] is not None:
        timestamp_check = results["timestamp_check"]
        print(
            f"  Timestamp pairs equal: {timestamp_check['equal_rows']:,}/"
            f"{timestamp_check['rows']:,}"
        )
    elapsed_difference = (
        results["full_read"]["elapsed_seconds"]
        - results["reduced_read"]["elapsed_seconds"]
    )
    print(f"  Reduced-read elapsed-time difference: {elapsed_difference:+.2f} s")


5EnvSensor_MayToDec2024_180kRows.csv
  full_read: 179,447 rows, 8 columns, 1.91 s, largest chunk 190.78 MiB
  reduced_read: 179,447 rows, 3 columns, 1.74 s, largest chunk 171.33 MiB
  Excluded columns: ['devicetype', 'processdate', 'processing_errors', 'sensorid', 'status']
  Timestamp pairs equal: 179,447/179,447
  Reduced-read elapsed-time difference: +0.17 s

5occupancySensor_MayToDec2024_9MRows.csv
  full_read: 9,091,732 rows, 8 columns, 16.07 s, largest chunk 43.64 MiB
  reduced_read: 9,091,732 rows, 5 columns, 12.06 s, largest chunk 23.17 MiB
  Excluded columns: ['deviceid', 'occupancystatus', 'previousoccupancystatus']
  Reduced-read elapsed-time difference: +4.01 s
